# 04_Dataset_Preparation.ipynb
Merge satellite features with ERA5 and MODIS into a single training dataset.

In [ ]:

!pip -q install pandas openpyxl

from google.colab import files
import pandas as pd

print("Upload features.csv")
up1 = files.upload()
features_file = list(up1.keys())[0]
features = pd.read_csv(features_file)

print("Upload ERA5 dataset CSV")
up2 = files.upload()
era5_file = list(up2.keys())[0]
era5 = pd.read_csv(era5_file)

print("Upload MODIS dataset CSV")
up3 = files.upload()
modis_file = list(up3.keys())[0]
modis = pd.read_csv(modis_file)

print("Upload Events dataset CSV (labels)")
up4 = files.upload()
events_file = list(up4.keys())[0]
events = pd.read_csv(events_file)

# Standardize keys
for df in [features, era5, modis, events]:
    if 'event_id' in df.columns:
        df['event_id'] = df['event_id'].astype(str)

# Merge
merged = features.merge(
    era5, on='event_id', how='left', suffixes=('','_era5')
).merge(
    modis, on='event_id', how='left', suffixes=('','_modis')
).merge(
    events[['event_id','event_type','event_class']],
    on='event_id',
    how='left'
)

# Create binary label (customize if needed)
merged['label'] = merged['event_type'].fillna('').str.contains(
    'Flood|Heat', case=False, regex=True
).astype(int)

merged.to_csv('training_dataset.csv', index=False)

print("\nTraining Dataset Shape:", merged.shape)
print(merged.head())

files.download('training_dataset.csv')
